# Per-pixel decomposition diagnostics

Run a decomposition, find the pixels the model reconstructs *worst*, and for each of them
open up **why** — three lenses:

1. **Loss landscape** — sweep each of the five material parameters
   (albedo R/G/B, roughness, metallic) one at a time, re-render the pixel under every
   image, and plot the data loss. Prediction and ground truth are marked, so you can see
   whether the fit sits in the true basin, a wrong local minimum, or a flat valley.
2. **Specular response** — the specular term alone over the same sweeps. This is usually
   where roughness/metallic are (un)identifiable: if the curve is flat, the data does not
   constrain that parameter.
3. **Gradient breakdown** — the data-loss gradient at the fitted point, split into the
   part flowing through the **diffuse** path and the part through the **specular** path,
   per parameter and per channel. Shows what the optimizer actually "feels" here.

Everything renders through the project's own `shade_ct_sh`, so the analysis is of exactly
the model that was fitted. Defaults to a self-contained synthetic scene (built with
`shade_ct_sh`, so it has exact GT and always runs); point `SCENE` at a dataset directory
to analyse a real run.


In [ ]:
import os, sys
from pathlib import Path

# Repo root, found by walking up from the CWD, then chdir so relative paths resolve
# whether Jupyter was launched from the repo root or from notebooks/.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "idr").is_dir())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)
os.environ.setdefault("WANDB_MODE", "disabled")

import numpy as np
import torch
import matplotlib.pyplot as plt

from idr.render import shade_ct_sh
from idr.render.brdf import _get_ggx_sh_lut
from idr.data.geometry import make_proxy_geometry
from idr.optim.registry import optimize

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("repo:", REPO, "| device:", DEVICE)


## 1. Configuration

`SCENE` is either `"synthetic"` (self-contained, exact GT) or a path to a dataset leaf
(`.../<view>/<dataset>/` holding `light_*.npy`, `sh_*.npy`, and the GT maps).

The decomposition config is an ordinary `cfg` dict — the same one `decompose_scene`
takes — so you can point this at LBFGS, LM, or VARPRO (incl. a `curriculum`).

In [ ]:
# ── what to decompose ─────────────────────────────────────────────────────────
SCENE      = "synthetic"          # "synthetic"  OR  a dataset-leaf path (string / Path)
N_IMAGES   = 24                   # observations to fit
DOWNSAMPLE = 8                    # only used for a real dataset (synthetic ignores it)

# ── how to decompose (any cfg decompose_scene accepts) ────────────────────────
from idr.config import DEFAULT_CFG
CFG = {**DEFAULT_CFG,
       "optimizer": "LBFGS", "n_iter": 60, "lbfgs_max_iter": 20,
       "loss": "L2", "double": False, "sh_order": 2,
       "tr_albedo": "sigmoid", "tr_metallic": "sigmoid", "tr_roughness": "sigmoid",
       "init_roughness_zero": True, "lambda_tv": 1e-5}
       # e.g. VarPro polish from a GD warm start:
       # "optimizer": "VARPRO", "n_iter": 20, "varpro_space": "natural",
       # "curriculum": [{"optimizer": "LBFGS", "n_iter": 200}],

# ── analysis knobs ────────────────────────────────────────────────────────────
N_WORST  = 4                      # how many worst-recon pixels to dissect
SWEEP_N  = 121                    # samples per parameter sweep
SEED     = 0                      # tie-break / reproducibility


## 2. Load the scene

Both sources are normalised into one structure: GT maps, the masked per-pixel geometry
`shade_ct_sh` consumes (`view`, `normal`, the GGX LUT), the observations `(n, M, 3)`, and
the GT SH lighting per image.

In [ ]:
def load_any(scene, n_images, downsample, device):
        # -> dict with GT maps, geometry, masked observations, and GT SH.
    if str(scene) == "synthetic":
        from tests.golden import build_scene
        sc = build_scene(device)
        maps = dict(normals=sc["normals"], mask=sc["mask"], albedo=sc["albedo"],
                    roughness=sc["roughness"], metallic=sc["metallic"])
        images = sc["images"][:n_images]
        sh_gt  = [np.asarray(s, np.float32) for s in sc["sh"][:n_images]]
    else:
        from idr.data.scene_io import load_scene
        s = load_scene(Path(scene), gt_npy=True)
        st = lambda a: np.ascontiguousarray(a[::downsample, ::downsample])
        maps = dict(normals=st(s["normals_np"]), mask=st(s["mask_np"]),
                    albedo=st(s["albedo_np"]), roughness=st(s["roughness_np"]),
                    metallic=st(s["metallic_np"]))
        images = [st(im) for im in s["images"][:n_images]]
        if s.get("sh_coeffs") is None:
            raise ValueError("scene has no GT sh_*.npy — needed to render observations")
        sh_gt = [np.asarray(c, np.float32) for c in s["sh_coeffs"][:n_images]]

    H, W = maps["normals"].shape[:2]
    Nhw, frag, mhw, cam = make_proxy_geometry(maps["normals"], maps["mask"], 60.0, 2.0,
                                              device, torch.float32)
    fm   = mhw.reshape(-1)
    N_m  = Nhw.reshape(-1, 3)[fm]
    V_m  = torch.nn.functional.normalize(cam[None] - frag.reshape(-1, 3)[fm], dim=-1)
    n_bands = int(CFG.get("sh_order", 2)) + 1
    lut  = _get_ggx_sh_lut(device, n_bands=n_bands).to(torch.float32)
    obs  = torch.stack([torch.from_numpy(im).to(device, torch.float32)
                        for im in images]).reshape(len(images), -1, 3)[:, fm, :]  # (n,M,3)
    sh_t = torch.from_numpy(np.stack(sh_gt)).to(device, torch.float32)            # (n,9|16,3)

    gt_px = {k: torch.from_numpy(maps[k].reshape(-1, maps[k].shape[-1] if maps[k].ndim == 3
                                                 else 1)).to(device, torch.float32)[fm]
             for k in ("albedo", "roughness", "metallic")}
    return dict(maps=maps, H=H, W=W, fm=fm, N_m=N_m, V_m=V_m, lut=lut, obs=obs,
                sh_t=sh_t, images=images, gt_px=gt_px,
                geom=(Nhw, frag, mhw, cam))

S = load_any(SCENE, N_IMAGES, DOWNSAMPLE, DEVICE)
print(f"{S['H']}x{S['W']}, {int(S['fm'].sum())} masked px, {S['obs'].shape[0]} images, "
      f"SH order {int(CFG.get('sh_order',2))}")


## 3. Run the decomposition

The recovered SH lighting is what the analysis re-lights with — the material sweeps below
hold lighting fixed at this estimate, exactly as the optimizer saw it.

In [ ]:
Nhw, frag, mhw, cam = S["geom"]
res = optimize("ct_sh", S["images"], Nhw, frag, mhw, cam,
               S["maps"]["metallic"], S["maps"]["roughness"], dict(CFG),
               gt_sh_coeffs=[c.cpu().numpy() for c in S["sh_t"]],
               gt_albedo=S["maps"]["albedo"])

fm = S["fm"]
est = {
    "albedo":    torch.from_numpy(res.albedo).to(DEVICE).reshape(-1, 3)[fm],       # (M,3)
    "roughness": torch.from_numpy(res.mat_b).to(DEVICE).reshape(-1, 1)[fm, 0],     # (M,)
    "metallic":  torch.from_numpy(res.mat_a).to(DEVICE).reshape(-1, 1)[fm, 0],     # (M,)
}
sh_est = torch.as_tensor(np.asarray(res.light), dtype=torch.float32, device=DEVICE)  # (n,9|16,3)
print("fitted:", {k: tuple(v.shape) for k, v in est.items()}, "| sh_est", tuple(sh_est.shape))


## 4. Reconstruction error → worst pixels

`render_px` is the one primitive everything else calls: it re-renders a single pixel
under **all** images for a given `(albedo, roughness, metallic)`, using the fitted
lighting. Because it is `shade_ct_sh`, the sweeps and gradients are of the true model.

Worst pixels are ranked by per-pixel RMSE over images at the fitted material.

In [ ]:
N_imgs = S["obs"].shape[0]

def render_px(pi, albedo, roughness, metallic, sh=None, want_spec=False):
        # Render pixel `pi` under every image. -> (n,3), or (recon, spec) if want_spec.
    sh = sh_est if sh is None else sh
    v, n = S["V_m"][pi:pi+1], S["N_m"][pi:pi+1]
    a = albedo[None] if albedo.ndim == 1 else albedo
    recon, spec = [], []
    for k in range(N_imgs):
        out = shade_ct_sh(v, n, a, sh[k], metallic[None, None], roughness[None, None],
                          lut=S["lut"], return_components=want_spec)
        if want_spec:
            out, comp = out
            spec.append(comp["spec"])
        recon.append(out)
    recon = torch.cat(recon, 0)                     # (n,3)
    return (recon, torch.cat(spec, 0)) if want_spec else recon

with torch.no_grad():
    M = int(fm.sum())
    err = torch.empty(M, device=DEVICE)
    for pi in range(M):
        r = render_px(pi, est["albedo"][pi], est["roughness"][pi], est["metallic"][pi])
        err[pi] = ((r - S["obs"][:, pi, :]) ** 2).mean().sqrt()

order = torch.argsort(err, descending=True)
worst = order[:N_WORST].tolist()
print("worst-pixel RMSE:", [round(float(err[p]), 4) for p in worst])

# error map (scatter back to image grid)
emap = np.full(S["H"] * S["W"], np.nan, np.float32)
emap[fm.cpu().numpy()] = err.cpu().numpy()
emap = emap.reshape(S["H"], S["W"])
ys, xs = np.divmod(fm.cpu().numpy()[worst], S["W"])

fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
im = ax[0].imshow(emap, cmap="inferno"); ax[0].set_title("per-pixel recon RMSE")
ax[0].scatter(xs, ys, s=80, facecolors="none", edgecolors="cyan", linewidths=1.8)
for i, (x, y) in enumerate(zip(xs, ys)):
    ax[0].annotate(f"#{i}", (x, y), color="cyan", fontsize=9,
                   xytext=(4, 4), textcoords="offset points")
plt.colorbar(im, ax=ax[0], fraction=0.046)
ax[1].hist(err.cpu().numpy(), bins=40, color="steelblue")
for p in worst: ax[1].axvline(float(err[p]), color="crimson", lw=1)
ax[1].set_title("RMSE distribution (worst marked)"); ax[1].set_xlabel("RMSE")
for a in ax[:1]: a.axis("off")
plt.tight_layout(); plt.show()


## 5. Loss landscape per parameter

For each worst pixel and each of the five parameters, sweep that parameter across its
range with the other four held at the fitted value, and plot the **total data loss**
(summed squared residual over all images) at that pixel.

Read it as: is the fit (dashed) at the bottom of the true basin (solid = GT)? A flat
curve means the data does not constrain that parameter here; two minima mean the pixel is
ambiguous; a fit away from a sharp GT minimum means the optimizer stalled.

In [ ]:
PARAMS = [("albedo R", 0), ("albedo G", 1), ("albedo B", 2),
          ("roughness", "roughness"), ("metallic", "metallic")]
RANGES = {"albedo R": (0.0, 1.0), "albedo G": (0.0, 1.0), "albedo B": (0.0, 1.0),
          "roughness": (0.03, 1.0), "metallic": (0.0, 1.0)}

def sweep_pixel(pi, name, key, grid, metric="loss"):
        # metric='loss' -> total data loss; 'spec' -> mean |specular| over images.
    a0 = est["albedo"][pi].clone(); r0 = est["roughness"][pi].clone(); m0 = est["metallic"][pi].clone()
    obs = S["obs"][:, pi, :]
    out = np.empty(len(grid), np.float32)
    with torch.no_grad():
        for j, val in enumerate(grid):
            a, r, m = a0.clone(), r0.clone(), m0.clone()
            if key in (0, 1, 2): a[key] = val
            elif key == "roughness": r = torch.tensor(float(val), device=DEVICE)
            else:                    m = torch.tensor(float(val), device=DEVICE)
            if metric == "loss":
                out[j] = float(((render_px(pi, a, r, m) - obs) ** 2).sum())
            else:
                _, sp = render_px(pi, a, r, m, want_spec=True)
                out[j] = float(sp.abs().mean())
    return out

def pred_gt(pi, name, key):
    if key in (0, 1, 2):
        return float(est["albedo"][pi][key]), float(S["gt_px"]["albedo"][pi][key])
    k = "roughness" if key == "roughness" else "metallic"
    return float(est[k][pi]), float(S["gt_px"][k][pi, 0])

def landscape_grid(metric, ylabel, title):
    fig, ax = plt.subplots(N_WORST, len(PARAMS), figsize=(3.0*len(PARAMS), 2.4*N_WORST),
                           squeeze=False)
    for i, pi in enumerate(worst):
        for j, (name, key) in enumerate(PARAMS):
            lo, hi = RANGES[name]; grid = np.linspace(lo, hi, SWEEP_N)
            y = sweep_pixel(pi, name, key, grid, metric=metric)
            pred, gt = pred_gt(pi, name, key)
            a = ax[i][j]
            a.plot(grid, y, color="steelblue", lw=1.6)
            a.axvline(pred, color="crimson", ls="--", lw=1.4, label="pred")
            a.axvline(gt,   color="green",   ls="-",  lw=1.4, label="GT")
            if i == 0: a.set_title(name, fontsize=10)
            if j == 0: a.set_ylabel(f"px #{i}\n{ylabel}", fontsize=8)
            a.tick_params(labelsize=7)
    ax[0][-1].legend(fontsize=8, loc="best")
    fig.suptitle(title, fontsize=12); plt.tight_layout(); plt.show()

landscape_grid("loss", "data loss", "Loss landscape per parameter  (dashed=pred, solid=GT)")


## 6. Specular term over the parameters

The same sweeps, but plotting only the **specular** contribution (mean `|spec|` over
images). This isolates where roughness and metallic act: the diffuse term barely moves
with them, so a flat specular curve means that parameter is unobservable at this pixel —
the loss landscape above will be flat there too, and the fit is then determined by the
prior/init rather than the data.

In [ ]:
landscape_grid("spec", "mean |spec|",
               "Specular magnitude per parameter  (dashed=pred, solid=GT)")


## 7. Gradient breakdown

At the fitted point, decompose the data-loss gradient. The loss is
`sum_k ||diff_k + spec_k - obs_k||^2`, and since the render is the *sum* of a diffuse and
a specular term, the gradient splits **additively** by path:

`d loss/d theta = sum_k 2 r_k · (d diff_k/d theta + d spec_k/d theta)`

with `r_k` the residual. Freezing `r_k` and back-propagating through the diffuse-only and
specular-only renders gives the two contributions exactly. This says *how the optimizer's
signal reaches each parameter*: a parameter whose gradient arrives almost entirely through
the specular path is only weakly identified whenever the specular term is small.

In [ ]:
def grad_split(pi):
        # -> dict param -> (g_total, g_diffuse_path, g_specular_path). Albedo entries are
    # per-channel 3-vectors; roughness/metallic are scalars.
    obs = S["obs"][:, pi, :]
    v, n = S["V_m"][pi:pi+1], S["N_m"][pi:pi+1]

    def render_terms(a, r, m):
        diff, spec = [], []
        for k in range(N_imgs):
            _, comp = shade_ct_sh(v, n, a[None], sh_est[k], m[None, None], r[None, None],
                                  lut=S["lut"], return_components=True)
            diff.append(comp["diff"]); spec.append(comp["spec"])
        return torch.cat(diff, 0), torch.cat(spec, 0)      # each (n,3)

    # residual at the fitted point, frozen
    with torch.no_grad():
        d0, s0 = render_terms(est["albedo"][pi], est["roughness"][pi], est["metallic"][pi])
        r_k = (d0 + s0) - obs                               # (n,3)

    def path_grad(term):  # term in {"diff","spec"}
        a = est["albedo"][pi].clone().requires_grad_(True)
        r = est["roughness"][pi].clone().requires_grad_(True)
        m = est["metallic"][pi].clone().requires_grad_(True)
        diff, spec = render_terms(a, r, m)
        comp = diff if term == "diff" else spec
        surrogate = (2.0 * r_k.detach() * comp).sum()       # d/dtheta = 2 r · dcomp/dtheta
        # allow_unused: roughness does NOT enter the diffuse term at all (it only acts
        # through the GGX specular lobe), so d(diff)/d(roughness) is exactly 0 and its
        # tensor is absent from the diffuse-only graph. A missing grad is a true zero.
        ga, gr, gm = torch.autograd.grad(surrogate, (a, r, m), allow_unused=True)
        z = lambda t: 0.0 if t is None else float(t)
        ga = np.zeros(3, np.float32) if ga is None else ga.detach().cpu().numpy()
        return ga, z(gr), z(gm)

    gd = path_grad("diff"); gs = path_grad("spec")
    out = {}
    for idx, nm in [(0, "albedo R"), (1, "albedo G"), (2, "albedo B")]:
        out[nm] = (gd[0][idx] + gs[0][idx], gd[0][idx], gs[0][idx])
    out["roughness"] = (gd[1] + gs[1], gd[1], gs[1])
    out["metallic"]  = (gd[2] + gs[2], gd[2], gs[2])
    return out

names = ["albedo R", "albedo G", "albedo B", "roughness", "metallic"]
fig, ax = plt.subplots(1, N_WORST, figsize=(3.4*N_WORST, 3.6), squeeze=False)
for i, pi in enumerate(worst):
    g = grad_split(pi)
    xloc = np.arange(len(names)); w = 0.4
    diff = [g[nm][1] for nm in names]; spec = [g[nm][2] for nm in names]
    a = ax[0][i]
    a.bar(xloc - w/2, diff, w, label="diffuse path", color="#4c78a8")
    a.bar(xloc + w/2, spec, w, label="specular path", color="#e45756")
    a.axhline(0, color="k", lw=0.6)
    a.set_xticks(xloc); a.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
    a.set_title(f"px #{i}  (RMSE {float(err[worst[i]]):.3f})", fontsize=9)
    if i == 0: a.set_ylabel("d(loss)/d(param)")
ax[0][0].legend(fontsize=8)
fig.suptitle("Gradient composition: diffuse vs specular path", fontsize=12)
plt.tight_layout(); plt.show()

# numeric table for the first worst pixel
print(f"pixel #0  (flat masked index {worst[0]}):")
g = grad_split(worst[0])
print(f"  {'param':10} {'total':>12} {'diffuse':>12} {'specular':>12}   pred / GT")
for nm, key in [("albedo R",0),("albedo G",1),("albedo B",2),
                ("roughness","roughness"),("metallic","metallic")]:
    tot, gdif, gsp = g[nm]
    pr, gt = pred_gt(worst[0], nm, key)
    print(f"  {nm:10} {tot:12.4e} {gdif:12.4e} {gsp:12.4e}   {pr:.3f} / {gt:.3f}")


### Reading the breakdown

- **Total near zero, both paths near zero** — converged and identifiable: the residual is
  small and there is no signal left to move on.
- **Total near zero, but the two paths are large and cancel** — a trade-off direction
  (classically albedo↔lighting scale, or roughness↔metallic through the specular lobe).
  The fit is at a saddle in that pair; the loss landscape shows the flat valley.
- **Gradient arrives almost entirely through the specular path** — that parameter is only
  as well-determined as the specular term is large. Cross-check with §6: if the specular
  magnitude is flat over that parameter, it is effectively unconstrained and the fit
  reflects the init/prior, not the data.
